# Fase 2 — Limpeza e Tratamento em Python

Este notebook formaliza, em Python, o tratamento de dados do projeto **Diagnóstico de Qualidade e Governança de Dados em Estabelecimentos de Saúde (CNES/DATASUS)**, recorte São Paulo, competência novembro/2025, fonte `basedosdados.br_ms_cnes` via Google BigQuery.

**Objetivo desta etapa:** a Fase 1 e a Fase 3 já validaram, em SQL, a estrutura da base e os indicadores de qualidade (completude, consistência, atualidade, unicidade). Este notebook não busca achado novo como objetivo principal — ele reproduz esses indicadores em Python, formalizando o pipeline de tratamento (leitura, padronização de tipos, tratamento de ausência, validação) que até aqui existia só como consultas SQL avulsas.

Na prática, essa reprodução funcionou também como uma segunda camada de verificação: duas divergências reais entre SQL e Python apareceram ao longo do processo (documentadas nas células correspondentes) e já foram corrigidas na documentação do projeto.

**Referências completas:**
- `docs/log-de-queries.md` — histórico de toda query SQL usada no projeto, com pergunta, resultado e decisão
- `docs/dicionario-de-dados.md` — achado final por campo, incluindo os dois corrigidos nesta Fase 2

In [1]:
from google.colab import auth
auth.authenticate_user()

## Leitura dos dados

Conexão direta ao BigQuery, em vez de exportar CSV manualmente. Essa escolha não é só preferência — um erro real desta sessão (uma contagem de linhas equivocada, causada por um arquivo CSV sem quebra de linha final) só existiu porque uma etapa anterior usou exportação manual. Ler direto da fonte elimina essa classe de erro.

Colunas selecionadas: a chave (`id_estabelecimento_cnes`), os cinco campos de completude já investigados em SQL, o campo auxiliar `tipo_grau_dependencia` (usado para a completude condicional de `cnpj_mantenedora`), e os dois campos de atualidade. Não é `SELECT *` — a tabela tem 204 colunas, e este notebook trabalha só com o subconjunto já validado.

In [2]:
from google.cloud import bigquery

client = bigquery.Client(project="SEU_PROJECT_ID")

query = """
SELECT COUNT(*) AS total_estabelecimentos
FROM `basedosdados.br_ms_cnes.estabelecimento`
WHERE sigla_uf = 'SP' AND ano = 2025 AND mes = 11
"""

resultado = client.query(query).to_dataframe()
resultado

,total_estabelecimentos
0,110362


In [3]:
query = """
SELECT
    id_estabelecimento_cnes,
    id_municipio,
    id_regiao_saude,
    tipo_unidade,
    tipo_gestao,
    cnpj_mantenedora,
    tipo_grau_dependencia,
    ano_atualizacao,
    mes_atualizacao
FROM `basedosdados.br_ms_cnes.estabelecimento`
WHERE sigla_uf = 'SP' AND ano = 2025 AND mes = 11
"""

df = client.query(query).to_dataframe()

In [4]:
print(df.shape)
df.dtypes

(110362, 9)


,0
id_estabelecimento_cnes,object
id_municipio,object
id_regiao_saude,object
tipo_unidade,object
tipo_gestao,object
cnpj_mantenedora,object
tipo_grau_dependencia,object
ano_atualizacao,Int64
mes_atualizacao,Int64


In [5]:
query_schema = """
SELECT column_name, data_type
FROM `basedosdados.br_ms_cnes.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'estabelecimento'
  AND column_name IN ('id_municipio', 'tipo_grau_dependencia')
"""

client.query(query_schema).to_dataframe()

,column_name,data_type
0,id_municipio,STRING
1,tipo_grau_dependencia,STRING


In [6]:
df.head()

,id_estabelecimento_cnes,id_municipio,id_regiao_saude,tipo_unidade,tipo_gestao,cnpj_mantenedora,tipo_grau_dependencia,ano_atualizacao,mes_atualizacao
0,7358512,3500709,206,22,M,,1,2025,2
1,2051885,3501202,215,2,M,46599817000129,3,2025,9
2,9194983,3501608,207,22,M,,1,2025,7
3,4564626,3501905,nan,22,M,,1,2024,4
4,7781547,3501905,207,22,M,,1,2024,7


In [7]:
df.isnull().sum()

,0
id_estabelecimento_cnes,0
id_municipio,0
id_regiao_saude,0
tipo_unidade,0
tipo_gestao,0
cnpj_mantenedora,0
tipo_grau_dependencia,0
ano_atualizacao,0
mes_atualizacao,0


## Tratamento de nulos

A investigação original em SQL já tinha revelado que `id_regiao_saude` não usa `NULL` para representar ausência — usa o texto literal `"nan"`. Uma checagem de `isnull()` sozinha, sem normalizar isso antes, mentiria: diria que o campo está 100% completo.

**Achado novo desta etapa:** normalizar as três formas de ausência de uma vez (`NULL`, `"nan"`, string vazia `""`) — em vez de só as duas já conhecidas — revelou 5 registros adicionais em `id_regiao_saude` com string vazia, nunca testados na investigação SQL original. Confirmado na fonte com uma query direta (`COUNT(*) WHERE id_regiao_saude = ''`), esse achado corrigiu a métrica oficial de completude de 60.361 (54,69%) para 60.366 (54,70%) — correção já aplicada em `docs/dicionario-de-dados.md`.

**Decisão para `id_regiao_saude`:** ausência mantida como `NA`, sem imputação e sem categoria `"não informado"`. O campo é um código territorial; inventar um valor ou uma categoria genérica alteraria o significado do dado original.

**Decisão para `cnpj_mantenedora`:** ausência também mantida como `NA`, mas com uma coluna auxiliar (`status_cnpj_mantenedora`) que preserva a distinção, já validada em SQL, entre ausência esperada (estabelecimentos "Individuais", `tipo_grau_dependencia = 1`) e ausência real (estabelecimentos "Mantidos" sem CNPJ, `tipo_grau_dependencia = 3`). Confirmado: zero casos de ausência real em 12.264 estabelecimentos "Mantidos".

In [8]:
import pandas as pd
import numpy as np

colunas_texto = ['id_municipio', 'id_regiao_saude', 'tipo_unidade',
                  'tipo_gestao', 'cnpj_mantenedora', 'tipo_grau_dependencia']

df[colunas_texto] = df[colunas_texto].replace(['nan', ''], pd.NA)

In [9]:
df.isnull().sum()

,0
id_estabelecimento_cnes,0
id_municipio,0
id_regiao_saude,60366
tipo_unidade,0
tipo_gestao,0
cnpj_mantenedora,98098
tipo_grau_dependencia,0
ano_atualizacao,0
mes_atualizacao,0


In [10]:
query_vazio = """
SELECT COUNT(*) AS total_string_vazia
FROM `basedosdados.br_ms_cnes.estabelecimento`
WHERE sigla_uf = 'SP' AND ano = 2025 AND mes = 11
  AND id_regiao_saude = ''
"""

client.query(query_vazio).to_dataframe()

,total_string_vazia
0,5


In [11]:
df[colunas_texto] = df[colunas_texto].astype('string')

## Padronização de tipos

**Achado desta etapa:** `id_municipio` estava documentado como `INT64`, mas a inspeção de `df.dtypes` mostrou `object` — tipo genérico demais para confirmar nada. Uma consulta direta a `INFORMATION_SCHEMA.COLUMNS` (célula anterior) confirmou o tipo real: `STRING`. Correção já aplicada em `docs/dicionario-de-dados.md`; as queries de SQL já tratavam o campo como texto, então nenhuma métrica de completude mudou — só a documentação estava errada.

Colunas de texto convertidas para o tipo `string` do pandas (não confundir com `str` nativo do Python) — mais restrito que `object`, aceita só texto e `pd.NA`, o que torna qualquer dado inesperado mais fácil de detectar no futuro.

In [12]:
df.dtypes

,0
id_estabelecimento_cnes,object
id_municipio,string[python]
id_regiao_saude,string[python]
tipo_unidade,string[python]
tipo_gestao,string[python]
cnpj_mantenedora,string[python]
tipo_grau_dependencia,string[python]
ano_atualizacao,Int64
mes_atualizacao,Int64


In [13]:
colunas_texto_completo = colunas_texto + ['id_estabelecimento_cnes']
df[colunas_texto_completo] = df[colunas_texto_completo].astype('string')

In [14]:
df['status_cnpj_mantenedora'] = 'preenchido'

df.loc[df['cnpj_mantenedora'].isna() & (df['tipo_grau_dependencia'] == '1'), 'status_cnpj_mantenedora'] = 'ausencia_esperada'
df.loc[df['cnpj_mantenedora'].isna() & (df['tipo_grau_dependencia'] == '3'), 'status_cnpj_mantenedora'] = 'ausencia_real'

df['status_cnpj_mantenedora'].value_counts()

,count
status_cnpj_mantenedora,
ausencia_esperada,98098
preenchido,12264


In [15]:
(df['status_cnpj_mantenedora'] == 'ausencia_real').sum()

np.int64(0)

In [16]:
df['id_estabelecimento_cnes'].duplicated().sum()

np.int64(0)

In [17]:
for coluna in colunas_texto_completo:
    total_com_espaco = (df[coluna].str.strip() != df[coluna]).sum()
    print(f"{coluna}: {total_com_espaco}")

id_municipio: 0
id_regiao_saude: 0
tipo_unidade: 0
tipo_gestao: 0
cnpj_mantenedora: 0
tipo_grau_dependencia: 0
id_estabelecimento_cnes: 0


## Validação dos campos

**Princípio orientador desta etapa:** domínio observado não é o mesmo que domínio válido. `tipo_gestao` mostrou só `M` e `E` neste recorte — mas isso não prova que outros códigos sejam inválidos, só que não ocorreram aqui. Por isso, cada validação abaixo usa como referência a tabela `dicionario` do próprio BigQuery (fonte primária do dataset), não suposição nem busca externa.

**Achado desta etapa:** a consulta ao `dicionario` revelou que `tipo_gestao` tem domínio oficial de 5 categorias (`M`, `E`, `D`, `S`, `Z`) — incluindo `Z` = "não informado", uma categoria que ninguém tinha cogitado antes. Também confirmou, para `tipo_grau_dependencia`, que `1` = "individual" e `3` = "mantida" — substituindo o que antes era só uma correspondência empírica sem nome oficial confirmado. As duas correções já estão em `docs/dicionario-de-dados.md`.

**Nota técnica:** testes de formato (`.str.len()`, `.str.startswith()`) só fazem sentido nos valores que existem — por isso `cnpj_mantenedora` é validado com `.dropna()` antes do teste, não na coluna inteira (que tem `NA` para os casos de ausência esperada).

In [18]:
tamanho_ok = (df['id_municipio'].str.len() == 7).all()
prefixo_ok = df['id_municipio'].str.startswith('35').all()

print(f"Todos com 7 dígitos: {tamanho_ok}")
print(f"Todos com prefixo 35: {prefixo_ok}")

Todos com 7 dígitos: True
Todos com prefixo 35: True


In [19]:
query_dicionario = """
SELECT chave, valor
FROM `basedosdados.br_ms_cnes.dicionario`
WHERE nome_coluna = 'tipo_gestao'
ORDER BY chave
"""

client.query(query_dicionario).to_dataframe()


,chave,valor
0,D,dupla
1,E,estadual
2,M,municipal
3,S,sem gestao
4,Z,nao informado


In [20]:
domain_tipo_gestao = {'D', 'E', 'M', 'S', 'Z'}

domain_ok = df['tipo_gestao'].isin(domain_tipo_gestao).all()
print(f"Todos os valores dentro do domínio oficial {domain_tipo_gestao}: {domain_ok}")

Todos os valores dentro do domínio oficial {'S', 'M', 'Z', 'D', 'E'}: True


In [21]:
cnpj_preenchidos = df['cnpj_mantenedora'].dropna()
tamanho_ok = (cnpj_preenchidos.str.len() == 14).all()

print(f"Total de CNPJs preenchidos avaliados: {len(cnpj_preenchidos)}")
print(f"Todos com 14 dígitos: {tamanho_ok}")

Total de CNPJs preenchidos avaliados: 12264
Todos com 14 dígitos: True


In [22]:
query_dicionario_grau = """
SELECT chave, valor
FROM `basedosdados.br_ms_cnes.dicionario`
WHERE nome_coluna = 'tipo_grau_dependencia'
ORDER BY chave
"""

client.query(query_dicionario_grau).to_dataframe()

,chave,valor
0,1,individual
1,3,mantida


In [23]:
domain_grau_dependencia = {'1', '3'}
domain_ok = df['tipo_grau_dependencia'].isin(domain_grau_dependencia).all()

print(f"Todos os valores dentro do domínio oficial {domain_grau_dependencia}: {domain_ok}")

Todos os valores dentro do domínio oficial {'1', '3'}: True


In [24]:
total = len(df)
ausentes = df['id_municipio'].isna().sum()
percentual_incompleto = round(ausentes / total * 100, 2)

print(f"Total: {total}")
print(f"Ausentes: {ausentes}")
print(f"% Incompleto: {percentual_incompleto}%")
print(f"Municípios distintos: {df['id_municipio'].nunique()}")

Total: 110362
Ausentes: 0
% Incompleto: 0.0%
Municípios distintos: 644


In [25]:
ausentes = df['id_regiao_saude'].isna().sum()
percentual_incompleto = round(ausentes / total * 100, 2)

print(f"Ausentes: {ausentes}")
print(f"% Incompleto: {percentual_incompleto}%")

Ausentes: 60366
% Incompleto: 54.7%


In [26]:
ausentes = df['tipo_unidade'].isna().sum()
percentual_incompleto = round(ausentes / total * 100, 2)

print(f"Ausentes: {ausentes}")
print(f"% Incompleto: {percentual_incompleto}%")

Ausentes: 0
% Incompleto: 0.0%


In [27]:
ausentes = df['tipo_gestao'].isna().sum()
percentual_incompleto = round(ausentes / total * 100, 2)

print(f"Ausentes: {ausentes}")
print(f"% Incompleto: {percentual_incompleto}%")
print()
print(df['tipo_gestao'].value_counts())

Ausentes: 0
% Incompleto: 0.0%

tipo_gestao
M    109733
E       629
Name: count, dtype: Int64


In [28]:
# Camada bruta
ausentes = df['cnpj_mantenedora'].isna().sum()
percentual_bruto = round(ausentes / total * 100, 2)

print(f"Completude bruta — Ausentes: {ausentes} ({percentual_bruto}%)")
print()

# Camada condicional
mantidos = df[df['tipo_grau_dependencia'] == '3']
ausentes_condicional = mantidos['cnpj_mantenedora'].isna().sum()
percentual_condicional = round(ausentes_condicional / len(mantidos) * 100, 2)

print(f"Completude condicional — Total 'mantidos': {len(mantidos)}")
print(f"Ausentes entre os 'mantidos': {ausentes_condicional} ({percentual_condicional}%)")

Completude bruta — Ausentes: 98098 (88.89%)

Completude condicional — Total 'mantidos': 12264
Ausentes entre os 'mantidos': 0 (0.0%)


In [29]:
import pandas as pd

comparacao = pd.DataFrame([
    {"indicador": "Completude id_municipio", "sql": "0 (0,0%)", "python": "0 (0.0%)", "situacao": "OK", "achado": "-"},
    {"indicador": "Completude id_regiao_saude", "sql": "60.361 (54,69%) — original", "python": "60.366 (54.7%)", "situacao": "DIVERGIU → corrigido", "achado": "Python revelou 5 registros com string vazia não testados no SQL original"},
    {"indicador": "Completude tipo_unidade", "sql": "0 (0,0%)", "python": "0 (0.0%)", "situacao": "OK", "achado": "-"},
    {"indicador": "Domínio tipo_gestao", "sql": "M=109.733, E=629", "python": "M=109733, E=629", "situacao": "OK", "achado": "Domínio oficial (dicionario) tem 5 categorias, só 2 usadas no recorte"},
    {"indicador": "Completude cnpj_mantenedora (bruta)", "sql": "98.098 (88,89%)", "python": "98098 (88.89%)", "situacao": "OK", "achado": "-"},
    {"indicador": "Completude cnpj_mantenedora (condicional)", "sql": "0 de 12.264 (0%)", "python": "0 de 12264 (0.0%)", "situacao": "OK", "achado": "Códigos 1/3 confirmados como individual/mantida via dicionario"},
    {"indicador": "Unicidade id_estabelecimento_cnes", "sql": "0 duplicados", "python": "0 duplicados", "situacao": "OK", "achado": "-"},
    {"indicador": "Tipo id_municipio", "sql": "documentado como INT64", "python": "STRING (confirmado)", "situacao": "DIVERGIU → corrigido", "achado": "Documentação estava errada; queries de SQL já tratavam como texto"},
])

comparacao

,indicador,sql,python,situacao,achado
0,Completude id_municipio,"0 (0,0%)",0 (0.0%),OK,-
1,Completude id_regiao_saude,"60.361 (54,69%) — original",60.366 (54.7%),DIVERGIU → corrigido,Python revelou 5 registros com string vazia nã...
2,Completude tipo_unidade,"0 (0,0%)",0 (0.0%),OK,-
3,Domínio tipo_gestao,"M=109.733, E=629","M=109733, E=629",OK,"Domínio oficial (dicionario) tem 5 categorias,..."
4,Completude cnpj_mantenedora (bruta),"98.098 (88,89%)",98098 (88.89%),OK,-
5,Completude cnpj_mantenedora (condicional),0 de 12.264 (0%),0 de 12264 (0.0%),OK,Códigos 1/3 confirmados como individual/mantid...
6,Unicidade id_estabelecimento_cnes,0 duplicados,0 duplicados,OK,-
7,Tipo id_municipio,documentado como INT64,STRING (confirmado),DIVERGIU → corrigido,Documentação estava errada; queries de SQL já ...
